# Einfluss von Finanznachrichten auf Währungspaare (Variante: eigenes kombiniertes CSV)

Zweite Variante des Analyse-Notebooks. **Unterschied zur ersten Version (`news_forex_korrelation.ipynb`):** Wir bauen zuerst aus den Rohdaten von Yahoo und EODHD ein **eigenes kombiniertes CSV** und arbeiten anschliessend nur noch mit dieser einen Datei. Das bereits existierende `forex_alle_quellen_kombiniert.csv` (aus `datenanalyse_forex.ipynb`) wird **nicht** verwendet.

MetaTrader wird wie in der ersten Version **ignoriert**. Die fünf Schritte (Quellenvergleich → Lückenfüllung → Quellen-Mittelwert → Sentiment-Overlay → Diskussion) sind identisch zur ersten Version.

## Setup und Aufbau des eigenen kombinierten CSV

Vorgehen:
1. Yahoo- und EODHD-Rohdaten für jedes Paar laden und auf das einheitliche Schema (`date`, `open`, `high`, `low`, `close`) bringen.
2. Pro Paar einen Outer-Join über das Datum machen (Spalten mit Quellen-Präfix: `yahoo_open`, `eodhd_open`, …).
3. Alle drei Paare in ein langes DataFrame zusammenfügen (Spalte `pair`).
4. Das Ergebnis als `data/processed/forex/forex_kombiniert_v2.csv` speichern und direkt wieder einlesen – ab dann arbeiten wir **nur noch mit dieser Datei**.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')

RAW_DIR       = Path('../../data/raw')
PROCESSED_DIR = Path('../../data/processed/forex')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PAIRS      = ['EUR_USD', 'EUR_CHF', 'GBP_USD']
OHLC_COLS  = ['open', 'high', 'low', 'close']
SOURCES    = ['yahoo', 'eodhd']
DATE_RANGE = '2022-01-01_to_2026-03-25'

OWN_COMBINED_CSV = PROCESSED_DIR / 'forex_kombiniert_v2.csv'


def _load_yahoo(pair: str) -> pd.DataFrame:
    path = RAW_DIR / 'forex' / 'yahoo' / f'{pair}_{DATE_RANGE}.csv'
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None).dt.normalize()
    df = df.rename(columns={'Open':'open','High':'high','Low':'low','Close':'close'})
    return df.set_index('date')[OHLC_COLS].sort_index()


def _load_eodhd(pair: str) -> pd.DataFrame:
    path = RAW_DIR / 'forex' / 'eodhd' / f'{pair}_{DATE_RANGE}.csv'
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date']).dt.normalize()
    return df.set_index('date')[OHLC_COLS].sort_index()


def build_combined_for_pair(pair: str) -> pd.DataFrame:
    """Outer-Join Yahoo + EODHD für ein Paar mit Quellen-Präfix in den Spaltennamen."""
    yh = _load_yahoo(pair).add_prefix('yahoo_')
    ed = _load_eodhd(pair).add_prefix('eodhd_')
    merged = yh.join(ed, how='outer').sort_index()
    merged['pair'] = pair
    return merged


# Schritt 1–3: alle Paare bauen und untereinander hängen
df_all = pd.concat([build_combined_for_pair(p) for p in PAIRS]).rename_axis('date')

# Schritt 4: speichern und wieder einlesen, damit ab hier wirklich nur noch mit dem CSV gearbeitet wird
df_all.to_csv(OWN_COMBINED_CSV)
df_all = pd.read_csv(OWN_COMBINED_CSV, parse_dates=['date']).set_index('date').sort_index()

print(f'Eigenes kombiniertes CSV gespeichert: {OWN_COMBINED_CSV}')
print(f'Zeilen: {len(df_all)} | Spalten: {list(df_all.columns)}')
df_all.head(3)

In [ ]:
def get_source(pair: str, source: str) -> pd.DataFrame:
    """Extrahiert für ein Paar die OHLC-Spalten einer Quelle aus dem eigenen kombinierten CSV.

    Zeilen, in denen die Quelle keine Werte hat, werden entfernt – wir erhalten so den
    natürlichen Datumsindex der jeweiligen Quelle.
    """
    cols = [f'{source}_{c}' for c in OHLC_COLS]
    sub = df_all.loc[df_all['pair'] == pair, cols].copy()
    sub.columns = OHLC_COLS
    return sub.dropna(how='all').sort_index()

raw_data = {pair: {src: get_source(pair, src) for src in SOURCES} for pair in PAIRS}
for pair, sources in raw_data.items():
    for src, df in sources.items():
        print(f'{pair:8s} {src:6s} | {len(df):5d} Zeilen | {df.index.min().date()} – {df.index.max().date()}')

## Schritt 1 — Quellenvergleich Yahoo vs. EODHD

Wir berechnen pro Paar und pro OHLC-Feld die rohe Differenz `yahoo - eodhd` für jedes gemeinsame Datum (in Kurspunkten, nicht in Prozent), schauen sie tabellarisch an, plotten sie über die Zeit und liefern danach eine prozentuale Übersicht.

In [ ]:
def compare_sources(pair: str) -> pd.DataFrame:
    """Inner-Join Yahoo vs. EODHD und berechne absolute + prozentuale Differenzen."""
    yh = raw_data[pair]['yahoo'].add_suffix('_yahoo')
    ed = raw_data[pair]['eodhd'].add_suffix('_eodhd')
    merged = yh.join(ed, how='inner')
    for col in OHLC_COLS:
        merged[f'{col}_diff_abs'] = merged[f'{col}_yahoo'] - merged[f'{col}_eodhd']
        merged[f'{col}_diff_pct'] = (merged[f'{col}_diff_abs'] / merged[f'{col}_eodhd']) * 100
    return merged

comparisons = {pair: compare_sources(pair) for pair in PAIRS}

# Absolute Differenzen pro Paar (nur die _diff_abs-Spalten, umbenannt)
diffs_abs = {
    pair: comparisons[pair][[f'{c}_diff_abs' for c in OHLC_COLS]].rename(
        columns={f'{c}_diff_abs': c for c in OHLC_COLS}
    )
    for pair in PAIRS
}
diffs_abs['EUR_USD'].head(10)

In [ ]:
# Plot der absoluten Differenzen pro Tag
fig, axes = plt.subplots(len(PAIRS), 1, figsize=(13, 9), sharex=True)
for ax, pair in zip(axes, PAIRS):
    df = diffs_abs[pair]
    for col in OHLC_COLS:
        ax.plot(df.index, df[col], label=col, alpha=0.7, linewidth=0.8)
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.6)
    ax.set_title(f'{pair} – Differenz pro Tag (Yahoo − EODHD), absolute Kurspunkte')
    ax.set_ylabel('Δ Kurs')
    ax.legend(loc='upper left', ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
# Aggregierte Übersicht in Prozent
summary_rows = []
for pair, df in comparisons.items():
    for col in OHLC_COLS:
        s = df[f'{col}_diff_pct'].abs()
        summary_rows.append({
            'pair': pair,
            'feld': col,
            'gemeinsame_tage': len(df),
            'mean_abw_%': s.mean(),
            'median_abw_%': s.median(),
            'max_abw_%': s.max(),
        })
pd.DataFrame(summary_rows)

In [ ]:
# Close-Verlauf beider Quellen pro Paar
fig, axes = plt.subplots(len(PAIRS), 1, figsize=(13, 9), sharex=True)
for ax, pair in zip(axes, PAIRS):
    ax.plot(raw_data[pair]['yahoo'].index, raw_data[pair]['yahoo']['close'], label='Yahoo', alpha=0.8)
    ax.plot(raw_data[pair]['eodhd'].index, raw_data[pair]['eodhd']['close'], label='EODHD', alpha=0.8)
    ax.set_title(f'{pair} – Close-Kurs Yahoo vs. EODHD')
    ax.set_ylabel('Kurs')
    ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Schritt 2 — Lückenfüllung durch lineare Interpolation

Wie in der ersten Version: Reindiziere jede Quelle pro Paar auf tägliche Frequenz und interpoliere fehlende Tage zeitgewichtet linear (`Mo=1, Mi=2 → Di=1.5`; `Mo=1, Do=3 → Di=1.667, Mi=2.333`).

Ausgabe: `data/processed/forex/{pair}_{quelle}_interpoliert_v2.csv` (Suffix `_v2`, damit die Dateien aus dem ersten Notebook nicht überschrieben werden).

In [ ]:
def fill_gaps(df: pd.DataFrame) -> pd.DataFrame:
    """Reindiziere auf tägliche Frequenz und interpoliere fehlende Tage zeitgewichtet linear."""
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='D')
    out = df.reindex(full_idx)
    out[OHLC_COLS] = out[OHLC_COLS].interpolate(method='time')
    out.index.name = 'date'
    return out

filled_data = {
    pair: {src: fill_gaps(df) for src, df in sources.items()}
    for pair, sources in raw_data.items()
}

for pair, sources in filled_data.items():
    for src, df in sources.items():
        n_orig   = len(raw_data[pair][src])
        n_filled = len(df)
        print(f'{pair:8s} {src:6s} | {n_orig} -> {n_filled} Zeilen ({n_filled - n_orig} interpoliert)')

In [ ]:
# Kontroll-Plot: ein kleiner Ausschnitt zeigt, dass die Lücken jetzt geschlossen sind
pair = 'EUR_USD'
orig   = raw_data[pair]['yahoo']['close'].loc['2024-02-01':'2024-02-29']
filled = filled_data[pair]['yahoo']['close'].loc['2024-02-01':'2024-02-29']

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(filled.index, filled.values, marker='o', label='interpoliert (täglich)', color='tab:orange')
ax.plot(orig.index,   orig.values,   marker='o', label='original (nur Handelstage)', color='tab:blue')
ax.set_title(f'{pair} – Beispiel-Ausschnitt Februar 2024 (Yahoo Close)')
ax.legend()
plt.tight_layout()
plt.show()

## Schritt 3 — Mittelwert über die Quellen (auf Basis der interpolierten Reihen)

Da Schritt 2 für beide Quellen denselben lückenlosen Tagesindex erzeugt hat, hat der Output von Schritt 3 keine Lücken. Pro Datum/Feld wird einfach `(yahoo + eodhd) / 2` gerechnet.

Anschliessend speichern wir **eine einzige finale Datei** im Long-Format mit allen drei Paaren: `data/processed/forex/forex_verarbeitet_v2.csv`.

In [ ]:
def average_sources(pair: str) -> pd.DataFrame:
    """Mittelwert OHLC über Yahoo + EODHD auf Basis der interpolierten Reihen aus Schritt 2.

    Da beide Quellen denselben lückenlosen Tagesindex haben, enthält der Output keine Lücken.
    """
    yh = filled_data[pair]['yahoo'][OHLC_COLS]
    ed = filled_data[pair]['eodhd'][OHLC_COLS]
    common = yh.index.intersection(ed.index)
    avg = (yh.loc[common] + ed.loc[common]) / 2
    avg.index.name = 'date'
    return avg.sort_index()

averaged_data = {pair: average_sources(pair) for pair in PAIRS}

# Ein einziges Long-Format-CSV mit allen Paaren – ersetzt die früheren Einzeldateien
df_processed = pd.concat(
    [df.assign(pair=pair) for pair, df in averaged_data.items()]
).rename_axis('date')
PROCESSED_CSV = PROCESSED_DIR / 'forex_verarbeitet_v2.csv'
df_processed.to_csv(PROCESSED_CSV)
print(f'Gespeichert: {PROCESSED_CSV} | {len(df_processed)} Zeilen')
df_processed.head()

## Schritt 3b — Diagnose des News-Sentiments

Bevor wir das Sentiment mit den Kursen kombinieren, schauen wir es uns genauer an:

1. Wie viele Artikel pro Tag haben wir überhaupt? Wie viele Tage fehlen ganz?
2. Wie sieht die Verteilung der einzelnen Artikel-`polarity`-Werte aus (ist die Mehrheit positiv oder negativ)?
3. Macht es einen grossen Unterschied, ob wir pro Tag mit **Mittelwert** oder **Median** aggregieren?
4. Wie stark streut das Sentiment innerhalb eines Tages?

**Hinweis:** Für **EUR_CHF** liefert EODHD im Datenbereich nur eine Handvoll Artikel – das Sentiment ist dort praktisch nicht aussagekräftig. Wir machen die Analyse trotzdem für alle drei Paare, der Vollständigkeit halber.

In [ ]:
import ast

PAIR_SYMBOL = {'EUR_USD':'EURUSD.FOREX','EUR_CHF':'EURCHF.FOREX','GBP_USD':'GBPUSD.FOREX'}

def _parse_list(val):
    if isinstance(val, list): return val
    if not isinstance(val, str) or not val.strip(): return []
    try: return ast.literal_eval(val)
    except (ValueError, SyntaxError): return []

raw_news = {}
for pair in PAIRS:
    df = pd.read_csv(RAW_DIR / 'news' / 'eodhd' / f'{pair}_news_{DATE_RANGE}.csv')
    df['date'] = pd.to_datetime(df['date_only'])
    df['symbols_list'] = df['symbols'].apply(_parse_list)
    df = df[df['symbols_list'].apply(lambda l: PAIR_SYMBOL[pair] in l)]
    df = df.dropna(subset=['polarity'])
    raw_news[pair] = df[['date', 'polarity']]

rows = []
for pair, df in raw_news.items():
    counts = df.groupby('date').size()
    full   = pd.date_range(df['date'].min(), df['date'].max(), freq='D') if not df.empty else []
    rows.append({'pair': pair,'artikel_total': len(df),'tage_mit_news': counts.shape[0],
                 'artikel_pro_tag_median': int(counts.median()) if len(counts) else 0,
                 'artikel_pro_tag_max': int(counts.max()) if len(counts) else 0,
                 'fehlende_tage': len(full) - counts.shape[0],
                 'fehlend_%': round((len(full) - counts.shape[0]) / max(len(full), 1) * 100, 1)})
pd.DataFrame(rows)

In [ ]:
# Verteilung der einzelnen Artikel-polarity pro Paar
fig, axes = plt.subplots(1, len(PAIRS), figsize=(15, 4))
for ax, pair in zip(axes, PAIRS):
    s = raw_news[pair]['polarity']
    if s.empty:
        ax.set_title(f'{pair} – keine Daten'); continue
    sns.histplot(s, bins=40, ax=ax, kde=True, color='tab:blue')
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
    ax.axvline(s.mean(),   color='tab:red',   linestyle='-',  label=f'Mean = {s.mean():.3f}')
    ax.axvline(s.median(), color='tab:green', linestyle='-',  label=f'Median = {s.median():.3f}')
    pos = (s > 0).mean() * 100
    neg = (s < 0).mean() * 100
    ax.set_title(f'{pair} – polarity (n={len(s)})\n{pos:.0f}% pos / {neg:.0f}% neg')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Mittelwert vs. Median pro Tag im Vergleich (Linienplot über die Zeit)
fig, axes = plt.subplots(len(PAIRS), 1, figsize=(13, 9), sharex=False)
for ax, pair in zip(axes, PAIRS):
    df = raw_news[pair]
    if df.empty:
        ax.set_title(f'{pair} – keine Daten'); continue
    g = df.groupby('date')['polarity']
    mean_s = g.mean()
    med_s  = g.median()
    ax.plot(mean_s.index, mean_s, label='Mittelwert', color='tab:red',   alpha=0.7, linewidth=0.8)
    ax.plot(med_s.index,  med_s,  label='Median',     color='tab:green', alpha=0.7, linewidth=0.8)
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.6)
    diff = (mean_s - med_s).abs()
    ax.set_title(f'{pair} – Tages-Aggregation Mean vs. Median  (|diff| median={diff.median():.3f}, max={diff.max():.3f})')
    ax.set_ylabel('polarity')
    ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Streuung innerhalb eines Tages: Boxplot der polarity-Werte für die letzten 30 Tage mit News (EUR_USD)
pair = 'EUR_USD'
df = raw_news[pair].copy()
last_days = df['date'].drop_duplicates().sort_values().tail(30)
subset = df[df['date'].isin(last_days)]
fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(data=subset, x='date', y='polarity', ax=ax, color='lightblue')
sns.stripplot(data=subset, x='date', y='polarity', ax=ax, color='black', size=2, alpha=0.5)
ax.axhline(0, color='red', linestyle='--', linewidth=0.7)
ax.set_title(f'{pair} – Streuung der Artikel-polarity innerhalb eines Tages (letzte 30 Tage mit News)')
ax.set_xticklabels([d.strftime('%Y-%m-%d') for d in last_days], rotation=60, ha='right')
plt.tight_layout()
plt.show()

### Empfehlung

Auf Basis der Diagnose:

- **Median statt Mittelwert** für die Tagesaggregation, sobald mindestens ein paar Artikel pro Tag vorliegen – er ist robuster gegen einzelne extreme Artikel und unterscheidet sich im Schnitt nur minimal vom Mittelwert.
- **Fehlende Tage** sind fast ausschliesslich Wochenenden/Feiertage (8–10 % bei EUR_USD/GBP_USD). Für die täglichen Plots lassen wir sie als NaN, das ist visuell ehrlich. Für **wöchentliche/monatliche** Aggregation füllen sie sich automatisch durch `resample().mean()`/`median()`.
- **Interpolation des Sentiments** macht meiner Meinung nach **wenig Sinn** (anders als beim Forex-Kurs): Sentiment ist keine kontinuierliche physikalische Grösse, die zwischen Messpunkten weiterläuft – an einem Wochenende erscheinen schlicht keine Artikel. Eine Interpolation würde künstliche Werte erzeugen.
- **EUR_CHF**: Zu wenige Artikel, um sinnvoll auszuwerten. Die Korrelation für dieses Paar ist nicht aussagekräftig.

Im nächsten Schritt wechseln wir die Aggregation auf **Median**.

## Schritt 4 — Kurs vs. News-Sentiment

Wie in der ersten Version: News pro Paar laden, **`polarity`** pro Tag mitteln, mit dem gemittelten Close joinen und in drei Auflösungen (täglich, wöchentlich, monatlich) als Overlay-Plot zeichnen. Zusätzlich Pearson-Korrelation pro Auflösung.

In [ ]:
# Kanonisches FOREX-Symbol pro Paar (für defensive Filterung der News)
PAIR_SYMBOL = {
    'EUR_USD': 'EURUSD.FOREX',
    'EUR_CHF': 'EURCHF.FOREX',
    'GBP_USD': 'GBPUSD.FOREX',
}

# Optional: nur Artikel berücksichtigen, deren tags-Liste **eine** dieser Tags enthält.
# Leeres Set/None = kein Tag-Filter (alle Artikel).
NEWS_TAG_FILTER: set[str] | None = None  # z. B. {'FX', 'MACRO', 'INFLATION'}

import ast

def _parse_list(val):
    """Wandelt einen String wie \"['FX','MACRO']\" in eine echte Python-Liste um."""
    if isinstance(val, list):
        return val
    if not isinstance(val, str) or not val.strip():
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []


def load_news(pair: str, tag_filter: set[str] | None = None) -> pd.DataFrame:
    """Lädt die EODHD-News für ein Paar und gibt eine tägliche Sentiment-Reihe zurück.

    - Filtert defensiv nach dem kanonischen Forex-Symbol des Paars (`symbols`-Spalte).
      So sind wir sicher, dass nur Artikel zählen, die wirklich zu diesem Paar gehören –
      auch wenn die Datei minimal andere Einträge enthält.
    - Optional zusätzlich nach `tags` filtern: ein Artikel zählt, wenn seine Tag-Liste
      mindestens **einen** der gewünschten Tags enthält.
    - Aggregation pro Tag: Median von `polarity` (robuster als Mittelwert, siehe Diagnose-Abschnitt 3b) (NaNs werden ignoriert).
    """
    path = RAW_DIR / 'news' / 'eodhd' / f'{pair}_news_{DATE_RANGE}.csv'
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date_only'])

    # 1) Symbolfilter (defensiv) – nur Artikel, die das kanonische FX-Symbol enthalten
    sym = PAIR_SYMBOL[pair]
    df['symbols_list'] = df['symbols'].apply(_parse_list)
    df = df[df['symbols_list'].apply(lambda l: sym in l)]

    # 2) Optionaler Tag-Filter
    if tag_filter:
        df['tags_list'] = df['tags'].apply(_parse_list)
        df = df[df['tags_list'].apply(lambda l: bool(set(l) & tag_filter))]

    return df.groupby('date')['polarity'].median().to_frame('sentiment').sort_index()


news_daily = {pair: load_news(pair, NEWS_TAG_FILTER) for pair in PAIRS}

def build_combined(pair: str) -> pd.DataFrame:
    fx  = averaged_data[pair][['close']].rename(columns={'close': 'close_avg'})
    nws = news_daily[pair]
    return fx.join(nws, how='outer').sort_index()

combined = {pair: build_combined(pair) for pair in PAIRS}
for pair, df in news_daily.items():
    print(f'{pair:8s} | {len(df):5d} Tage mit News (nach Filter)')
combined['EUR_USD'].tail()

In [ ]:
def aggregate(df: pd.DataFrame, freq: str) -> pd.DataFrame:
    """Aggregiert (close_avg + sentiment) per Mittelwert auf 'D' / 'W-MON' / 'MS'."""
    if freq == 'D':
        return df.copy()
    return df.resample(freq).mean()

def plot_overlay(pair: str):
    df = combined[pair]
    settings = [
        ('D',     'Täglich'),
        ('W-MON', 'Wöchentlich (Mittelwert pro Woche)'),
        ('MS',    'Monatlich (Mittelwert pro Monat)'),
    ]
    fig, axes = plt.subplots(len(settings), 1, figsize=(13, 10))
    for ax, (freq, label) in zip(axes, settings):
        agg = aggregate(df, freq)
        l1, = ax.plot(agg.index, agg['close_avg'], color='tab:blue', label=f'{pair} Close (Mittelwert Yahoo+EODHD)')
        ax.set_ylabel('Kurs', color='tab:blue')
        ax.tick_params(axis='y', labelcolor='tab:blue')
        ax2 = ax.twinx()
        l2, = ax2.plot(agg.index, agg['sentiment'], color='tab:red', alpha=0.7, label=f'Sentiment {pair} News (polarity)')
        ax2.axhline(0, color='grey', linestyle='--', linewidth=0.7)
        ax2.set_ylabel('Sentiment (polarity)', color='tab:red')
        ax2.tick_params(axis='y', labelcolor='tab:red')
        ax.set_title(f'{pair} – {label}')
        ax.legend([l1, l2], [l1.get_label(), l2.get_label()], loc='upper left')
    plt.tight_layout()
    plt.show()

for pair in PAIRS:
    plot_overlay(pair)

In [ ]:
# Pearson-Korrelation Kurs vs. Sentiment auf den drei Auflösungen
rows = []
for pair in PAIRS:
    for freq, label in [('D', 'täglich'), ('W-MON', 'wöchentlich'), ('MS', 'monatlich')]:
        agg = aggregate(combined[pair], freq).dropna()
        corr = agg['close_avg'].corr(agg['sentiment']) if len(agg) > 2 else np.nan
        rows.append({'pair': pair, 'auflösung': label, 'n': len(agg), 'pearson_r': corr})
pd.DataFrame(rows)

## Schritt 4b — Ölpreise dazunehmen

Rohöl (WTI und Brent) hat einen bekannten Einfluss auf Währungen wie USD, CAD, GBP. Wir laden die Yahoo-Ölpreis-CSVs aus `data/raw/oil/yahoo/`, bringen sie auf das gleiche Schema, interpolieren Lücken (gleiche Methode wie in Schritt 2) und plotten dann **drei Reihen übereinander** pro Paar:

1. Forex-Close (Mittelwert beider Quellen, links)
2. News-Sentiment (rechts, rot, gestrichelte Nulllinie)
3. Ölpreis-Close (rechts, grün, eigene Skala mit kleinem Offset)

Alles in den drei Auflösungen täglich / wöchentlich / monatlich, jeweils mit dem Mittelwert aggregiert.

In [ ]:
def load_oil(name: str, file_stem: str) -> pd.DataFrame:
    """Lädt ein Yahoo-Öl-CSV (gleiche Spaltenstruktur wie Yahoo-Forex) und interpoliert Lücken."""
    path = RAW_DIR / 'oil' / 'yahoo' / f'{file_stem}_{DATE_RANGE}.csv'
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None).dt.normalize()
    df = df.rename(columns={'Open':'open','High':'high','Low':'low','Close':'close'})
    df = df.set_index('date')[OHLC_COLS].sort_index()
    return fill_gaps(df)  # gleiche Interpolation wie bei Forex

oil_data = {
    'WTI':   load_oil('WTI',   'WTI_Crude_Oil'),
    'Brent': load_oil('Brent', 'Brent_Crude_Oil'),
}
for name, df in oil_data.items():
    print(f'{name:6s} | {len(df):5d} Tage | {df.index.min().date()} – {df.index.max().date()}')
oil_data['WTI'].head(3)

In [ ]:
def build_combined_with_oil(pair: str, oil_name: str) -> pd.DataFrame:
    """Kombiniert Forex-Close (Mittelwert), Sentiment und Ölpreis-Close auf einem gemeinsamen Datumsindex."""
    fx  = averaged_data[pair][['close']].rename(columns={'close': 'close_avg'})
    nws = news_daily[pair]
    oil = oil_data[oil_name][['close']].rename(columns={'close': f'oil_{oil_name.lower()}'})
    return fx.join(nws, how='outer').join(oil, how='outer').sort_index()


def plot_overlay_with_oil(pair: str, oil_name: str = 'Brent'):
    """Drei Auflösungen: Forex (links), Sentiment + Öl (rechts auf je eigener Skala)."""
    df = build_combined_with_oil(pair, oil_name)
    oil_col = f'oil_{oil_name.lower()}'
    settings = [
        ('D',     'Täglich'),
        ('W-MON', 'Wöchentlich (Mittelwert pro Woche)'),
        ('MS',    'Monatlich (Mittelwert pro Monat)'),
    ]
    fig, axes = plt.subplots(len(settings), 1, figsize=(13, 11))
    for ax, (freq, label) in zip(axes, settings):
        agg = df.copy() if freq == 'D' else df.resample(freq).mean()
        # 1. Forex-Close (linke Achse)
        l1, = ax.plot(agg.index, agg['close_avg'], color='tab:blue', label=f'{pair} Close (Mittelwert Quellen)')
        ax.set_ylabel('Forex-Kurs', color='tab:blue')
        ax.tick_params(axis='y', labelcolor='tab:blue')
        # 2. Sentiment (rechte Achse 1)
        ax2 = ax.twinx()
        l2, = ax2.plot(agg.index, agg['sentiment'], color='tab:red', alpha=0.6, label=f'Sentiment {pair} News (polarity)')
        ax2.axhline(0, color='grey', linestyle='--', linewidth=0.6)
        ax2.set_ylabel('Sentiment', color='tab:red')
        ax2.tick_params(axis='y', labelcolor='tab:red')
        # 3. Öl (rechte Achse 2, nach aussen versetzt)
        ax3 = ax.twinx()
        ax3.spines['right'].set_position(('outward', 55))
        l3, = ax3.plot(agg.index, agg[oil_col], color='tab:green', alpha=0.7, label=f'{oil_name} Crude Oil Close')
        ax3.set_ylabel(f'{oil_name} (USD)', color='tab:green')
        ax3.tick_params(axis='y', labelcolor='tab:green')
        ax.set_title(f'{pair} vs. Sentiment vs. {oil_name} – {label}')
        ax.legend([l1, l2, l3], [l1.get_label(), l2.get_label(), l3.get_label()], loc='upper left')
    plt.tight_layout()
    plt.show()

for pair in PAIRS:
    plot_overlay_with_oil(pair, oil_name='Brent')

In [ ]:
# Korrelationsmatrix Forex / Sentiment / Öl auf Tagesbasis pro Paar
for pair in PAIRS:
    df = build_combined_with_oil(pair, 'Brent').dropna()
    print(f'\n{pair} – Pearson-Korrelation (täglich, n={len(df)}):')
    print(df.corr().round(3))

## Schritt 5 — Diskussion

Die Forex-Ergebnisse sind identisch zur ersten Version, weil dieselben Rohdaten über einen anderen Lade-Pfad fliessen. Zusätzlich haben wir hier den **Ölpreis** (WTI / Brent) als dritte Reihe in die Overlay-Plots aufgenommen.

Hinweise:
- Im Beispiel-Plot wird **Brent** verwendet (`plot_overlay_with_oil(pair, 'Brent')`); für WTI einfach `'WTI'` übergeben.
- Korrelationen auf den **Niveaus** sind nur ein erster Hinweis; für eine ernsthafte Auswertung sollte man Renditen vergleichen oder Lag-Korrelationen prüfen.

**Ausgaben dieses Notebooks (nur 2 Dateien):**
- `forex_kombiniert_v2.csv` – Rohdaten Yahoo + EODHD pro Paar in einem Long-Format-CSV
- `forex_verarbeitet_v2.csv` – interpoliert + Quellen-Mittelwert OHLC, alle Paare in einem Long-Format-CSV